In [2]:
#!/usr/bin/env python3
"""
Full single-file pipeline:
1) Search for trained fold runs and locate their results.csv and best.pt.
2) Select best fold checkpoint by highest metrics/mAP50-95(M) (fallbacks applied).
3) Load best model and run model.val() to compute metrics (saves into metrics.save_dir).
4) Compute/print/save precision/recall/f1 via confusion matrix and per-class tables.
5) Run qualitative predictions on one image per resolution (10/20/40/60/80 cm) and save side-by-side plots.
6) Save plots (confusion matrix, per-class PRF, per-class mAP) and export a submission JSON from predictions.
"""
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import json
import shutil
from datetime import datetime
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import os
from ultralytics import YOLO
import torch

# --------- CONFIG ----------
ROOT_DIR = Path('..')                # adjust if your repo root differs
DATA_DIR = ROOT_DIR / 'data' / 'processed'
VAL_IMAGES_DIR = DATA_DIR / 'images' / 'val'
VAL_LABELS_DIR = DATA_DIR / 'labels' / 'val'
MODEL_CONFIG_PATH = ROOT_DIR / 'configurations' / 'model_data-seg.yaml'
SEARCH_RUN_DIRS = [
    ROOT_DIR / 'notebooks'
]
RESULTS_CANDIDATE_PREFIX = 'YOLO'    # optional filter: only consider run folders containing this
OUTPUT_ROOT = ROOT_DIR / 'best_model_eval'
CONF_THRESHOLD = 0.25
RESOLUTIONS = ['10cm', '20cm', '40cm', '60cm', '80cm']
# --------------------------

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = OUTPUT_ROOT / f"best_model_eval_{timestamp}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Utility to try multiple candidate metric column names (most common)
METRIC_KEYS_CANDIDATES = [
    'metrics/mAP50-95(M)',
    'metrics/mAP50-95(M)_',
    'metrics/mAP50-95(M).0',
    'metrics/mAP50-95(M) ', 
    'metrics/mAP50-95(M)\r',
    'metrics/mAP50-95(M)\n',
    'metrics/mAP50-95(M)_seg',
    'metrics/mAP50-95(M) ', 
    'metrics/mAP50-95(B)',  # fallback to box metric if mask missing
    'metrics/mAP50(M)',     # fallback
    'metrics/mAP50-95',     # generic fallback
]

# --------- 1) discover runs and fold candidates ----------
print("Searching for candidate runs with results.csv and best.pt ...")
candidates = []  # list of dicts: {fold_dir, results_csv, best_pt, run_name}

for base in SEARCH_RUN_DIRS:
    if not base.exists():
        continue
    # walk a couple of levels to find fold_* directories with results.csv and weights/best.pt
    for run_dir in base.iterdir():
        # optionally filter by prefix
        try:
            if not run_dir.is_dir():
                continue
        except Exception:
            continue

        # look for fold_* children or run_dir itself being a fold
        # Case A: run_dir/fold_0/results.csv
        folds = list(run_dir.glob('fold_*'))
        if not folds:
            # maybe the run_dir itself is fold_0 style
            folds = [run_dir] if (run_dir / 'results.csv').exists() else []

        for fold_dir in folds:
            results_csv = fold_dir / 'results.csv'
            # typical weights location: fold_dir/weights/best.pt
            best_pt = fold_dir / 'weights' / 'best.pt'
            # some setups write best.pt at fold_dir/weights/best.pt OR fold_dir/weights/best.pt
            if results_csv.exists() and best_pt.exists():
                candidates.append({
                    'fold_dir': fold_dir,
                    'results_csv': results_csv,
                    'best_pt': best_pt,
                    'run_root': run_dir
                })

# If no candidates found, try searching deeper recursively
if not candidates:
    for base in SEARCH_RUN_DIRS:
        if not base.exists(): 
            continue
        for fold_results in base.rglob('results.csv'):
            fold_dir = fold_results.parent
            # find best.pt under fold_dir or parent weights
            best_pt_candidates = list(fold_dir.rglob('best.pt'))
            if best_pt_candidates:
                candidates.append({
                    'fold_dir': fold_dir,
                    'results_csv': fold_results,
                    'best_pt': best_pt_candidates[0],
                    'run_root': fold_dir.parent
                })

if not candidates:
    print("No candidate runs found. Looked in:", SEARCH_RUN_DIRS)
    sys.exit(1)

print(f"Found {len(candidates)} candidate folds.")

# --------- 2) read results.csv and extract metric (last epoch row) ----------
def extract_metric_from_results(csv_path, candidate_keys=METRIC_KEYS_CANDIDATES):
    """Load results.csv and return (value, used_key) or (None, None)"""
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"[WARN] Failed to read {csv_path}: {e}")
        return (None, None)
    if df.empty:
        return (None, None)
    # last epoch row (YOLO's results.csv has epoch per row)
    last_row = df.iloc[-1]
    cols = df.columns.tolist()
    # try candidate keys
    for k in candidate_keys:
        if k in cols:
            try:
                return (float(last_row[k]), k)
            except Exception:
                continue
    # fallback: try to find any column containing "mAP" and "(M)" (mask)
    mask_candidates = [c for c in cols if ('mAP' in c or 'map' in c) and '(M)' in c]
    if mask_candidates:
        c = mask_candidates[0]
        try:
            return (float(last_row[c]), c)
        except Exception:
            pass
    # fallback: find any column with 'mAP' and pick largest semantically
    generic = [c for c in cols if 'mAP' in c or 'map' in c]
    if generic:
        c = generic[0]
        try:
            return (float(last_row[c]), c)
        except Exception:
            pass
    # else None
    return (None, None)

# collect metrics for each candidate
for cand in candidates:
    val, used = extract_metric_from_results(cand['results_csv'])
    cand['metric_value'] = val
    cand['metric_key'] = used
    print(f"Candidate {cand['fold_dir']} -> metric {used} = {val}")

# Filter out candidates with None metric_value, but still allow fallback using mAP50(M)
candidates_with_metric = [c for c in candidates if c['metric_value'] is not None]
if not candidates_with_metric:
    print("[WARN] No candidates had a recognized metric column. Trying fallback using any results.csv's mAP50(M) or mAP columns.")
    # try to salvage any numeric column
    for c in candidates:
        df = pd.read_csv(c['results_csv'])
        # attempt to find numeric columns and pick 'metrics/mAP50(M)' if present
        for key in df.columns:
            if 'mAP50' in key and '(M)' in key:
                try:
                    val = float(df.iloc[-1][key])
                    c['metric_value'] = val
                    c['metric_key'] = key
                    candidates_with_metric.append(c)
                    break
                except Exception:
                    continue
if not candidates_with_metric:
    print("Still no usable metric columns found. Exiting.")
    sys.exit(1)

# choose best candidate by max metric_value
best = max(candidates_with_metric, key=lambda x: x['metric_value'] if x['metric_value'] is not None else -1)
print("\nSelected BEST candidate:")
print("  fold_dir:", best['fold_dir'])
print("  best_pt:", best['best_pt'])
print("  metric key:", best['metric_key'])
print("  metric value:", best['metric_value'])

# copy selected best weights to out_dir for traceability
selected_weights_copy = OUT_DIR / 'best_fold_weights.pt'
shutil.copy(best['best_pt'], selected_weights_copy)
print(f"Copied best weights to {selected_weights_copy}")

# --------- 3) load model and run validation (model.val) ----------
print("\nLoading model and running model.val() ... This may take a while.")
weights_path = str(best['best_pt'])
model = YOLO(weights_path)

# Create a data config path (use user config)
if not MODEL_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Model data config not found: {MODEL_CONFIG_PATH}")
data_config = str(MODEL_CONFIG_PATH)

# Run validation and save results to disk (metrics.save_dir)
with torch.inference_mode():
    metrics = model.val(data=data_config, imgsz=640, conf=CONF_THRESHOLD)

# metrics.save_dir location
metrics_save_dir = Path(metrics.save_dir)
print("Validation outputs saved to:", metrics_save_dir)

# copy a single config file for provenance
try:
    shutil.copy(MODEL_CONFIG_PATH, metrics_save_dir / MODEL_CONFIG_PATH.name)
except Exception:
    pass

# --------- 4) Compute precision/recall/f1 from confusion matrix ----------
print("\nComputing precision/recall/f1 from confusion matrix (if available)...")
if hasattr(metrics, "confusion_matrix") and metrics.confusion_matrix is not None:
    cm = metrics.confusion_matrix.matrix.astype(np.int64)  # (nc+1, nc+1)
    names = metrics.names if isinstance(metrics.names, (list, tuple)) else list(metrics.names.values())
    nc = len(names)
    tp = np.diag(cm)[:nc].astype(np.int64)
    fp = cm[:nc, :].sum(axis=1) - tp
    fn = cm[:, :nc].sum(axis=0) - tp
    precision = tp / (tp + fp + 1e-16)
    recall = tp / (tp + fn + 1e-16)
    f1 = 2 * precision * recall / (precision + recall + 1e-16)
    prf_df = pd.DataFrame({
        'class': names,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })
    prf_df.to_csv(metrics_save_dir / 'P&R_results.csv', index=False)
    print(f"Saved P&R table to {metrics_save_dir/'P&R_results.csv'}")
else:
    print("[WARN] metrics.confusion_matrix not available on metrics object.")
    prf_df = pd.DataFrame()

# also export metrics summary and per-epoch results
try:
    df_all = metrics.to_df()
    df_all.to_csv(metrics_save_dir / 'validation_results_full.csv', index=False)
except Exception:
    pass

# save summary_dict
try:
    summary_df = pd.DataFrame([metrics.results_dict])
    summary_df.to_csv(metrics_save_dir / 'summary_metrics_results.csv', index=False)
    print("Saved summary metrics.")
except Exception:
    pass

# --------- 5) Visualizations (confusion matrix heatmap, per-class PRF, per-class mAP) ----------
print("\nGenerating plots ...")
sns.set(style="whitegrid")
# confusion heatmap (TP/FP/FN)
try:
    if not prf_df.empty:
        cm_df = prf_df.set_index('class')[['TP', 'FP', 'FN']]
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title("TP / FP / FN per Class")
        plt.tight_layout()
        plt.savefig(metrics_save_dir / 'confusion_TP_FP_FN.png')
        plt.close()
        # PRF bar chart
        metrics_plot = prf_df.set_index('class')[['precision', 'recall', 'f1']]
        ax = metrics_plot.plot(kind='bar', figsize=(10, 6))
        ax.set_ylim(0, 1)
        ax.set_ylabel("Score")
        plt.title("Per-Class Precision, Recall, F1")
        plt.tight_layout()
        plt.savefig(metrics_save_dir / 'per_class_PRF.png')
        plt.close()
except Exception as e:
    print("Plot error:", e)

# per-class mAP if available in metrics.seg.maps
try:
    seg_metrics = getattr(metrics, 'seg', None) or getattr(metrics, 'mask', None) or getattr(metrics, 'task', None)
    # metrics may expose seg metrics via metrics.seg
    seg = getattr(metrics, 'seg', None)
    map_vals = None
    if seg is not None and hasattr(seg, "maps") and seg.maps is not None:
        map_vals = seg.maps
    elif hasattr(metrics, 'maps') and metrics.maps is not None:
        map_vals = metrics.maps
    elif hasattr(metrics, 'results_dict') and 'per_class' in metrics.results_dict:
        # try to read per-class from results_dict
        per = metrics.results_dict.get('per_class', {})
        map_vals = [per.get(k, np.nan) for k in range(len(names))]
    if map_vals is not None:
        map_df = pd.DataFrame({'class': names, 'mAP50-95': map_vals})
        plt.figure(figsize=(6, 4))
        sns.barplot(data=map_df, x='class', y='mAP50-95')
        plt.ylim(0, 1)
        plt.title("Per-Class mAP@0.50:0.95")
        plt.tight_layout()
        plt.savefig(metrics_save_dir / 'per_class_mAP.png')
        plt.close()
except Exception:
    pass

# copy metrics folder to OUT_DIR for consolidated outputs
shutil.copytree(metrics_save_dir, OUT_DIR / metrics_save_dir.name)

# --------- 6) Select 1 image per resolution and make qualitative comparisons ----------
print("\nSelecting one image per resolution and producing qualitative comparison images ...")
val_images = sorted([p for p in VAL_IMAGES_DIR.glob('*.tif')])
selected_images = []
for res in RESOLUTIONS:
    found = False
    for img_path in val_images:
        if img_path.name.startswith(res):
            selected_images.append(img_path)
            found = True
            break
    if not found:
        print(f"[WARN] No image found starting with {res} in {VAL_IMAGES_DIR}")

print("Selected images:", [p.name for p in selected_images])

# helper: draw ground truth from text label (YOLO segmentation text format: class x1 y1 x2 y2 ... normalized)
def draw_ground_truth(image_path):
    label_path = Path(str(image_path).replace('images', 'labels').replace('.tif', '.txt'))
    image = cv2.imread(str(image_path))
    if image is None:
        raise RuntimeError(f"Failed to read {image_path}")
    h, w = image.shape[:2]
    if not label_path.exists():
        return image
    with open(label_path, 'r') as f:
        lines = f.readlines()
    colors = [(255, 0, 0), (0, 0, 255), (0,255,0), (255,255,0)]
    overlay = image.copy()
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        try:
            class_id = int(parts[0])
        except:
            continue
        coords = [float(x) for x in parts[1:]]
        if len(coords) < 6:
            continue
        # coords are normalized, transform to pixel pairs
        pts = np.array(coords).reshape(-1, 2)
        pts[:,0] = np.clip(pts[:,0] * w, 0, w-1)
        pts[:,1] = np.clip(pts[:,1] * h, 0, h-1)
        pts_int = pts.astype(np.int32)
        color = colors[class_id % len(colors)]
        cv2.fillPoly(overlay, [pts_int], color)
    alpha = 0.45
    out = cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0)
    return out

# plotting function using the loaded model
def plot_comparison(image_path, model, out_path):
    gt_img = draw_ground_truth(image_path)
    # prediction
    preds = model.predict(source=str(image_path), conf=CONF_THRESHOLD, verbose=False)
    # results[0].plot returns numpy rgb image (Ultralytics)
    try:
        pred_img = preds[0].plot(boxes=False, labels=False)
    except Exception:
        # fallback: read saved prediction image if any
        pred_img = cv2.imread(str(image_path))
    # combine and save
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    ax1.imshow(cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB))
    ax1.set_title('Ground Truth')
    ax1.axis('off')
    ax2.imshow(cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB))
    ax2.set_title(f'Prediction: {image_path.name}')
    ax2.axis('off')
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()
    print("Saved comparison:", out_path)

qual_out_dir = OUT_DIR / 'qualitative'
qual_out_dir.mkdir(parents=True, exist_ok=True)
for img_path in selected_images:
    out_path = qual_out_dir / f"{img_path.stem}_comparison.png"
    try:
        plot_comparison(img_path, model, out_path)
    except Exception as e:
        print("Failed qualitative for", img_path, "err:", e)

# --------- 7) Run predictions on full val set and export annotations (txt -> json) ----------
print("\nRunning predictions on full validation set and exporting annotation-like JSON ...")
pred_out_dir = OUT_DIR / 'predictions'
pred_out_dir.mkdir(parents=True, exist_ok=True)
# run prediction and save predicted txts
with torch.inference_mode():
    # predict(..., save=True, save_txt=True) will save predicted .txt files per image into the save_dir
    preds = model.predict(source=str(VAL_IMAGES_DIR), conf=CONF_THRESHOLD,
                          save=True, save_txt=True,
                          project=str(pred_out_dir), name='pred', exist_ok=True, verbose=False)

# locate prediction txts saved by ultralytics: path structure: pred_out_dir/pred/labels/<image_name>.txt
labels_dir = pred_out_dir / 'pred' / 'labels'
if not labels_dir.exists():
    # sometimes predictions saved under predictions/pred/labels
    possible = list(pred_out_dir.glob('**/labels'))
    if possible:
        labels_dir = possible[0]
print("Predicted labels dir:", labels_dir)

# parse predicted txts into JSON (similar to your earlier functions)
def parse_pred_txts(labels_dir, imgs_dir):
    imgs = []
    annotations = []
    for img_p in sorted(Path(imgs_dir).glob('*.tif')):
        with ImageOpenWarningSuppress():
            pass
    # list prediction txts
    for txt in sorted(labels_dir.glob('*.txt')):
        image_name = txt.name
        # map txt to tif name
        tif_name = image_name[:-4] + '.tif'
        # read file
        with open(txt, 'r') as f:
            for line in f:
                toks = line.strip().split()
                if len(toks) < 3:
                    continue
                cls_id = int(float(toks[0]))
                coords = [float(x) for x in toks[1:]]
                # some save formats include confidence as last field; detect if odd number of coords
                # if odd length, assume last is confidence and drop it for segmentation
                if len(coords) % 2 == 1:
                    # drop last value (confidence)
                    coords = coords[:-1]
                seg_pairs = [(coords[i], coords[i+1]) for i in range(0, len(coords), 2)]
                annotations.append({
                    'image': tif_name,
                    'class': int(cls_id),
                    'segmentation_norm': [list(p) for p in seg_pairs]
                })
    # make images meta
    image_metas = []
    for img_p in sorted(Path(VAL_IMAGES_DIR).glob('*.tif')):
        from PIL import Image
        with Image.open(img_p) as im:
            w, h = im.size
        image_metas.append({
            'file_name': img_p.name,
            'width': w,
            'height': h,
            'annotations': []
        })
    # append annotations denormalized
    img_map = {m['file_name']: m for m in image_metas}
    for a in annotations:
        fn = a['image']
        if fn not in img_map:
            continue
        w = img_map[fn]['width']; h = img_map[fn]['height']
        seg_norm = a['segmentation_norm']
        seg_px = []
        for x_norm, y_norm in seg_norm:
            X = int(round(max(0, min(w-1, x_norm * w))))
            Y = int(round(max(0, min(h-1, y_norm * h))))
            seg_px.extend([X, Y])
        img_map[fn]['annotations'].append({
            'class': int(a['class']),
            'segmentation': seg_px
        })
    return image_metas

# a tiny helper to suppress PIL warnings used above
class ImageOpenWarningSuppress:
    def __enter__(self):
        return None
    def __exit__(self, exc_type, exc_val, exc_tb):
        return False

pred_json = parse_pred_txts(labels_dir, VAL_IMAGES_DIR)
OUT_JSON = OUT_DIR / 'predictions_submission.json'
with open(OUT_JSON, 'w') as f:
    json.dump({'images': pred_json}, f, indent=2)
print("Saved prediction JSON to:", OUT_JSON)

# --------- 8) Copy a few provenance files and conclude ----------
# copy best weights, fold info, and metrics folder to OUT_DIR (already copied metrics folder)
try:
    shutil.copy(selected_weights_copy, OUT_DIR / selected_weights_copy.name)
except Exception:
    pass

# Done
print("\n==== ALL DONE ====")
print("Outputs saved to:", OUT_DIR)
print("Qualitative previews:", qual_out_dir)
print("Prediction artifacts:", pred_out_dir)
print("Validation metrics saved under the metrics folder copy inside the OUT_DIR")


Searching for candidate runs with results.csv and best.pt ...
Found 10 candidate folds.
Candidate ..\notebooks\YOLOv11m_CV\fold_0 -> metric metrics/mAP50-95(M) = 0.14769
Candidate ..\notebooks\YOLOv11m_CV\fold_1 -> metric metrics/mAP50-95(M) = 0.15925
Candidate ..\notebooks\YOLOv11m_CV\fold_2 -> metric metrics/mAP50-95(M) = 0.14125
Candidate ..\notebooks\YOLOv11m_CV\fold_3 -> metric metrics/mAP50-95(M) = 0.14799
Candidate ..\notebooks\YOLOv11m_CV\fold_4 -> metric metrics/mAP50-95(M) = 0.14818
Candidate ..\notebooks\YOLOv11s_CV\fold_0 -> metric metrics/mAP50-95(M) = 0.13932
Candidate ..\notebooks\YOLOv11s_CV\fold_1 -> metric metrics/mAP50-95(M) = 0.15473
Candidate ..\notebooks\YOLOv11s_CV\fold_2 -> metric metrics/mAP50-95(M) = 0.11748
Candidate ..\notebooks\YOLOv11s_CV\fold_3 -> metric metrics/mAP50-95(M) = 0.14803
Candidate ..\notebooks\YOLOv11s_CV\fold_4 -> metric metrics/mAP50-95(M) = 0.13404

Selected BEST candidate:
  fold_dir: ..\notebooks\YOLOv11m_CV\fold_1
  best_pt: ..\notebook

val: Scanning C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\data\processed\labels\val.cache... 30 images, 0 bac
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP


                   all         30       7960      0.598      0.291      0.442      0.251      0.535      0.263      0.399      0.195
       individual_tree         30       7104      0.729      0.331      0.543      0.312      0.594      0.269      0.448      0.223
        group_of_trees         26        856      0.466      0.251      0.341      0.189      0.475      0.256       0.35      0.166
Speed: 0.7ms preprocess, 31.3ms inference, 0.0ms loss, 4.8ms postprocess per image
Results saved to C:\Users\hmanasi1\Documents\ADML\Project\runs\segment\val8
Validation outputs saved to: C:\Users\hmanasi1\Documents\ADML\Project\runs\segment\val8

Computing precision/recall/f1 from confusion matrix (if available)...
Saved P&R table to C:\Users\hmanasi1\Documents\ADML\Project\runs\segment\val8\P&R_results.csv
Saved summary metrics.

Generating plots ...

Selecting one image per resolution and producing qualitative comparison images ...
Selected images: ['10cm_train_19.tif', '20cm_train_38.tif', 